# Environment Setup

**Purpose**
Prepare the local training environment and install the Diffusers package used by this environment.

**Outputs**
- Installed Python dependencies
- Installed `diffusers` package in the active Python environment
- Local copy of the official LoRA training script in `training/`


In [ ]:
# Environment Setup: install training dependencies and prepare the Diffusers training script used by this environment
# This step only prepares the environment and does not start training.

import sys
import subprocess
from pathlib import Path
import urllib.request

# Resolve directories. The notebook can be run from the repository root or from training/.
cwd = Path.cwd()
if (cwd / "training").exists():
    project_dir = cwd
elif cwd.name == "training":
    project_dir = cwd.parent
else:
    raise RuntimeError("Run this notebook from the Inkward_Bound repository root or from training/.")

training_dir = project_dir / "training"
train_script = training_dir / "train_text_to_image_lora.py"

# Minimal training dependency set. This notebook prioritizes stability and does not install xformers or bitsandbytes by default.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "diffusers"], check=False)

packages = [
    "diffusers",
    "accelerate",
    "transformers",
    "datasets",
    "peft",
    "ftfy",
    "tensorboard",
    "safetensors",
    "sentencepiece",
    "huggingface_hub",
    "matplotlib",
    "pillow",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *packages], check=True)

import importlib
import site
importlib.invalidate_caches()
site.main()

try:
    diffusers = importlib.import_module("diffusers")
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "diffusers import failed. Rerun Environment Setup once more; if it still fails, restart the kernel and rerun from the top."
    ) from exc

# Download the official training script matching the installed diffusers version.
diffusers_version = getattr(diffusers, "__version__", "main")
candidate_urls = [
    f"https://raw.githubusercontent.com/huggingface/diffusers/v{diffusers_version}/examples/text_to_image/train_text_to_image_lora.py",
    "https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py",
]

downloaded = False
for url in candidate_urls:
    try:
        urllib.request.urlretrieve(url, train_script)
        downloaded = True
        print("training script =", url)
        break
    except Exception:
        continue

if not downloaded:
    if train_script.exists():
        print("Download failed; using the existing copy at", train_script)
    else:
        raise RuntimeError("Could not download train_text_to_image_lora.py and no local copy exists.")

print("project_dir =", project_dir)
print("diffusers =", diffusers_version)


# Hardware Check

**Purpose**
Verify local hardware readiness before training.

**Checks**
- CUDA / MPS availability
- Active GPU model
- Available VRAM

**Expectation**
This project trains on Apple Silicon (MPS) or an NVIDIA GPU. CPU-only training is possible but slow.


In [ ]:
# Hardware Check: inspect the local training environment and GPU status

import platform
import torch
import subprocess

print("python =", platform.python_version())
print("pytorch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())
print("mps available =", torch.backends.mps.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("gpu =", gpu_name)
    print("vram_gb =", round(total_memory_gb, 2))
    try:
        subprocess.run(["nvidia-smi"], check=False)
    except FileNotFoundError:
        print("nvidia-smi not found, skip.")
elif torch.backends.mps.is_available():
    print("Apple Silicon MPS detected. Training runs locally but slower than CUDA.")
else:
    print("Warning: no GPU detected. Local training may be too slow.")


# Dataset Validation

**Purpose**
Convert the ink dataset into the Diffusers `imagefolder` format and validate it before training.

**Inputs**
- `ink_dataset/01_pure_diffusion` … `04_gathering_ink` (101 JPEGs with same-name `.txt` captions)

**Caption structure**
`inkwb, <state phrase>, <phase phrase>, <viewpoint>, <morphology>, monochrome, high contrast`

**Outputs**
- `training/dataset/images/` (resized copies, max 1024 px)
- `training/dataset/metadata.jsonl`
- Validation summary and sample preview


In [ ]:
# Dataset Validation: convert ink_dataset captions to metadata.jsonl and verify the training data
# prepare_dataset.py holds the conversion logic so the command line and this notebook stay in sync.

import json
import random
import subprocess
import matplotlib.pyplot as plt
from PIL import Image

dataset_dir = training_dir / "dataset"
metadata_path = dataset_dir / "metadata.jsonl"

# v4 experiment: trim abstract phase phrases and style tags at dataset-build time.
# Set to False to rebuild the full v3-style captions. Source .txt files are never modified.
TRIM_CAPTIONS = True

cmd = [sys.executable, str(training_dir / "prepare_dataset.py")]
if TRIM_CAPTIONS:
    cmd.append("--trim")
subprocess.run(cmd, check=True, cwd=project_dir)

records = [json.loads(line) for line in metadata_path.read_text(encoding="utf-8").splitlines() if line.strip()]

missing_images = [r["file_name"] for r in records if not (dataset_dir / r["file_name"]).exists()]
bad_captions = [r["file_name"] for r in records if not r["text"].startswith("inkwb, ")]
if missing_images or bad_captions:
    raise ValueError({"missing_images": missing_images, "bad_captions": bad_captions})

categories = {}
for r in records:
    cat = r["file_name"].split("/")[-1].rsplit("_", 1)[0]
    categories[cat] = categories.get(cat, 0) + 1

print("records =", len(records))
for cat, n in sorted(categories.items()):
    print(f"  {cat}: {n}")

samples = random.sample(records, 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, rec in zip(axes, samples):
    ax.imshow(Image.open(dataset_dir / rec["file_name"]), cmap="gray")
    ax.set_title(rec["text"][:60] + "…", fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()


# Training Configuration

**Purpose**
Define the training configuration for the `inkwb` ink-diffusion LoRA run and generate the exact launch command.

**Outputs**
- `training/runs/inkwb_lora_v3/training_config.json`
- Final `accelerate launch` command
- Step count estimate for the current dataset size
- Training log file path


In [ ]:
# Training Configuration: set hyperparameters and build the training command
# This step only configures the run and does not start training.
# Hyperparameters are carried over from The-Latent-Mycelium mycelium_lora_structure_v1 (80 images, successful run).
# random_flip stays disabled because the ink captions encode left/right positions.

import math
import json

base_model = "runwayml/stable-diffusion-v1-5"
trigger_phrase = "inkwb"
validation_prompt = (
    "inkwb, black ink gathering and condensing in water, side view, "
    "heavy ink covering most of the frame, a dense black mound with a twisting tendril column above"
)
seed = 42

runs_dir = training_dir / "runs"
output_dir = runs_dir / "inkwb_lora_v4"
preview_dir = output_dir / "preview"
log_path = output_dir / "training.log"
for d in (runs_dir, output_dir, preview_dir):
    d.mkdir(parents=True, exist_ok=True)

train_config = {
    "pretrained_model_name_or_path": base_model,
    "train_data_dir": str(dataset_dir),
    "output_dir": str(output_dir),
    "resolution": 512,
    "center_crop": True,
    "train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 12,
    "learning_rate": 5e-5,
    "lr_scheduler": "constant",
    "lr_warmup_steps": 0,
    "rank": 16,
    "checkpointing_steps": 100,
    "validation_epochs": 2,
    "num_validation_images": 2,
    "dataloader_num_workers": 0,
    "report_to": "tensorboard",
    "mixed_precision": "fp16" if torch.cuda.is_available() else "no",
    "resume_from_checkpoint": None,
}

update_steps_per_epoch = math.ceil(len(records) / train_config["train_batch_size"] / train_config["gradient_accumulation_steps"])
estimated_total_steps = update_steps_per_epoch * train_config["num_train_epochs"]

train_command = [
    sys.executable, "-m", "accelerate.commands.launch",
    "--num_processes", "1",
    "--num_machines", "1",
    "--mixed_precision", train_config["mixed_precision"],
    "--dynamo_backend", "no",
    str(train_script),
    "--pretrained_model_name_or_path", train_config["pretrained_model_name_or_path"],
    "--train_data_dir", train_config["train_data_dir"],
    "--caption_column", "text",
    "--resolution", str(train_config["resolution"]),
    "--center_crop",
    "--train_batch_size", str(train_config["train_batch_size"]),
    "--gradient_accumulation_steps", str(train_config["gradient_accumulation_steps"]),
    "--num_train_epochs", str(train_config["num_train_epochs"]),
    "--learning_rate", str(train_config["learning_rate"]),
    "--lr_scheduler", train_config["lr_scheduler"],
    "--lr_warmup_steps", str(train_config["lr_warmup_steps"]),
    "--rank", str(train_config["rank"]),
    "--seed", str(seed),
    "--checkpointing_steps", str(train_config["checkpointing_steps"]),
    "--validation_prompt", validation_prompt,
    "--validation_epochs", str(train_config["validation_epochs"]),
    "--num_validation_images", str(train_config["num_validation_images"]),
    "--dataloader_num_workers", str(train_config["dataloader_num_workers"]),
    "--report_to", train_config["report_to"],
    "--output_dir", train_config["output_dir"],
]

config_payload = {
    "trigger_phrase": trigger_phrase,
    "validation_prompt": validation_prompt,
    "seed": seed,
    "log_path": str(log_path),
    "image_count": len(records),
    "update_steps_per_epoch": update_steps_per_epoch,
    "estimated_total_steps": estimated_total_steps,
    **train_config,
}
(output_dir / "training_config.json").write_text(json.dumps(config_payload, indent=2), encoding="utf-8")

print("images =", len(records))
print("update_steps_per_epoch =", update_steps_per_epoch)
print("estimated_total_steps =", estimated_total_steps)
print("output_dir =", output_dir)
print()
print(" ".join(map(str, train_command)))


# Start Training

**Purpose**
Launch the LoRA training job with the official script running against the Diffusers package installed in this environment.

**Notes**
- This is the longest-running stage in the notebook
- Outputs are written to `training/runs/inkwb_lora_v4`
- Full stdout and stderr are also written to `training.log`


In [ ]:
# Start Training: launch the official LoRA training flow
# Before running, confirm that Environment Setup, Hardware Check, Dataset Validation, and Training Configuration all completed.

import os
import subprocess
from pathlib import Path

print("train_script =", train_script)
print("output_dir =", output_dir)
print("log_path =", log_path)

if not Path(train_script).exists():
    raise FileNotFoundError(f"Training script not found at {train_script}. Rerun Environment Setup.")

train_env = os.environ.copy()

with log_path.open("w", encoding="utf-8") as log_file:
    log_file.write("Training command:\n")
    log_file.write(" ".join(map(str, train_command)) + "\n\n")
    log_file.flush()

    process = subprocess.Popen(
        train_command,
        env=train_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    raise subprocess.CalledProcessError(return_code, train_command)


# Inference Test

**Purpose**
Load the trained LoRA weights and evaluate whether the caption vocabulary is steerable: state phrases, phase phrases, and viewpoint.

**Prompt groups**
- `baseline`: trigger word with minimal description
- `state_control`: the four system states (diffusing / layered / disturbed / gathering)
- `phase_control`: one state swept across early → final phase (the c-value axis)
- `viewpoint_control`: top-down view vs side view

**Outputs**
- Preview images under `training/runs/inkwb_lora_v4/preview`
- Side-by-side comparison grids


In [ ]:
# Inference Test: load the trained LoRA weights and run a steerability evaluation

import torch
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32
print("device =", device)

STYLE = "monochrome, high contrast"
# Photographic qualities are requested positively instead of arriving via container leakage.
PHOTO = "macro photograph, wet glossy ink, soft light"
# Narrow negatives: only concrete container features. Broad words like glass/reflection
# also suppress the wet photographic look of dense ink, so they are excluded here.
# WATER anchors the in-water context that v1 got from container leakage; without it
# the model drifts toward ink-on-paper. Add "water surface line at the top" when wanted.
WATER = "suspended in clear water"
NEG = "tank walls, basin rim, bubbles, table edge, dark smears at the bottom edge, paper texture, ink on paper, photo border, dark frame edges"
prompt_groups = {
    "baseline": [
        f"inkwb, black ink {WATER}, {PHOTO}, {STYLE}",
        f"inkwb, ink diffusing in water, soft billowing plume, {WATER}, {PHOTO}, {STYLE}",
    ],
    "state_control": [
        f"inkwb, black ink diffusing freely across still water, developing phase of diffusion, top-down view, {PHOTO}, {STYLE}",
        f"inkwb, layered black ink suspended in water, developing phase of settling, side view, {WATER}, {PHOTO}, {STYLE}",
        f"inkwb, turbulent agitated black ink in water, developing phase of disturbance, side view, {WATER}, {PHOTO}, {STYLE}",
        f"inkwb, black ink gathering and condensing in water, developing phase of gathering, side view, {WATER}, {PHOTO}, {STYLE}",
    ],
    # Phase words are paired with the measured density phrases from measure_ink_coverage.py —
    # these exact phrases exist in the training captions, giving the temporal axis a visual anchor.
    "phase_control": [
        f"inkwb, black ink gathering and condensing in water, early phase of gathering, side view, {WATER}, sparse ink traces, mostly clear water, {PHOTO}, {STYLE}",
        f"inkwb, black ink gathering and condensing in water, developing phase of gathering, side view, {WATER}, ink spreading across part of the frame, {PHOTO}, {STYLE}",
        f"inkwb, black ink gathering and condensing in water, advanced phase of gathering, side view, {WATER}, dense ink covering much of the frame, {PHOTO}, {STYLE}",
        f"inkwb, black ink gathering and condensing in water, final phase of gathering, side view, {WATER}, heavy ink covering most of the frame, {PHOTO}, {STYLE}",
    ],
    "viewpoint_control": [
        f"inkwb, black ink diffusing freely across still water, developing phase of diffusion, top-down view, {PHOTO}, {STYLE}",
        f"inkwb, black ink diffusing freely across still water, developing phase of diffusion, side view, {WATER}, {PHOTO}, {STYLE}",
    ],
}

pipe = StableDiffusionPipeline.from_pretrained(
    base_model,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.load_lora_weights(str(output_dir), weight_name="pytorch_lora_weights.safetensors")
pipe = pipe.to(device)

num_inference_steps = 28
guidance_scale = 7.5
index = 0
for group, prompts in prompt_groups.items():
    images = []
    for prompt in prompts:
        torch.manual_seed(seed)
        image = pipe(
            prompt=prompt,
            negative_prompt=NEG,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            width=512,
            height=512,
        ).images[0]
        index += 1
        image.save(preview_dir / f"{index:02d}_{group}.png")
        images.append((prompt, image))

    fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4.6))
    if len(images) == 1:
        axes = [axes]
    for ax, (prompt, image) in zip(axes, images):
        ax.imshow(image)
        ax.set_title(prompt.replace("inkwb, ", "")[:70], fontsize=7)
        ax.axis("off")
    fig.suptitle(group)
    plt.tight_layout()
    plt.show()

# Evaluation checklist:
# - Does inkwb reproduce the monochrome ink-in-water look?
# - Do the four state phrases produce distinct morphologies?
# - Do early -> final phase prompts move along a plausible temporal axis?
# - Does top-down view vs side view switch the camera angle?


# Evaluation Matrix

**Purpose**
Generate the acceptance-test grid: 4 states × 4 phases per seed, plus a seed-diversity row. This is the pass/fail gate before batch-generating the latent atlas.

**Acceptance criteria (in order of importance)**
1. Steerability — the four state rows look distinct, and each row changes visibly from early to final phase
2. Clean style — no container features unless prompted
3. Diversity — the same prompt across different seeds gives varied compositions
4. No overfitting — outputs are not copies of the training photographs

**Outputs**
- One 4×4 grid image per seed and one diversity strip under `preview/eval/`


In [ ]:
# Evaluation Matrix: 4 states x 4 phases per seed + seed-diversity strip
# Reuses the pipeline loaded in the Inference Test cell. Runtime note: each seed grid
# is 16 images; with 3 seeds this is 48 generations. Reduce eval_seeds to [42] for a quick pass.

import torch
import matplotlib.pyplot as plt

eval_seeds = [42, 123, 777]
eval_dir = preview_dir / "eval"
eval_dir.mkdir(parents=True, exist_ok=True)

STATES = [
    ("diffusion",   "black ink diffusing freely across still water", "diffusion",   "top-down view"),
    ("settling",    "layered black ink suspended in water",          "settling",    "side view"),
    ("disturbance", "turbulent agitated black ink in water",         "disturbance", "side view"),
    ("gathering",   "black ink gathering and condensing in water",   "gathering",   "side view"),
]
# Each phase is paired with the measured density phrase used in the v3 captions.
PHASES = [
    ("early", "sparse ink traces, mostly clear water"),
    ("developing", "ink spreading across part of the frame"),
    ("advanced", "dense ink covering much of the frame"),
    ("final", "heavy ink covering most of the frame"),
]

def build_prompt(state_phrase, process, phase, density, viewpoint):
    water = f"{WATER}, " if viewpoint == "side view" else ""
    return f"inkwb, {state_phrase}, {phase} phase of {process}, {viewpoint}, {water}{density}, {PHOTO}, {STYLE}"

def generate(prompt, s):
    torch.manual_seed(s)
    return pipe(
        prompt=prompt,
        negative_prompt=NEG,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        width=512,
        height=512,
    ).images[0]

# 1. State x phase grid per seed
for s in eval_seeds:
    fig, axes = plt.subplots(4, 4, figsize=(16, 17))
    for row, (label, state_phrase, process, viewpoint) in enumerate(STATES):
        for col, (phase, density) in enumerate(PHASES):
            img = generate(build_prompt(state_phrase, process, phase, density, viewpoint), s)
            ax = axes[row][col]
            ax.imshow(img)
            ax.axis("off")
            if row == 0:
                ax.set_title(f"{phase} phase", fontsize=10)
            if col == 0:
                ax.text(-0.08, 0.5, label, transform=ax.transAxes, fontsize=10,
                        rotation=90, va="center", ha="center")
    fig.suptitle(f"state x phase matrix, seed {s}", fontsize=13)
    plt.tight_layout()
    fig.savefig(eval_dir / f"matrix_seed_{s}.png", dpi=120)
    plt.show()

# 2. Seed-diversity strip: one fixed prompt across all seeds
div_prompt = build_prompt(STATES[3][1], STATES[3][2], PHASES[1][0], PHASES[1][1], STATES[3][3])
fig, axes = plt.subplots(1, len(eval_seeds), figsize=(4 * len(eval_seeds), 4.4))
for ax, s in zip(axes, eval_seeds):
    img = generate(div_prompt, s)
    img.save(eval_dir / f"diversity_seed_{s}.png")
    ax.imshow(img)
    ax.set_title(f"seed {s}", fontsize=10)
    ax.axis("off")
fig.suptitle("seed diversity, fixed prompt (gathering / developing)", fontsize=12)
plt.tight_layout()
fig.savefig(eval_dir / "diversity_strip.png", dpi=120)
plt.show()

# Scoring guide:
# 1. Steerability: rows distinct AND columns progress -> pass
# 2. Clean style: no tank walls / rims / surface lines in any cell -> pass
# 3. Diversity: the three diversity images differ in composition -> pass
# 4. Overfitting: spot-check cells against ink_dataset originals -> pass if not near-copies
# All four pass -> proceed to atlas batch generation. Phase columns flat -> revisit training.


# Result Packaging

**Purpose**
Package the final training artifacts for local backup, transfer, or later reuse.

**Outputs**
- Zip archive next to the training run directory
- File listing of saved checkpoints and generated assets


In [ ]:
# Result Packaging: export and compress the local training outputs

import zipfile

output_files = sorted([p for p in output_dir.rglob("*") if p.is_file()])
print("output file count =", len(output_files))

for path in output_files[:30]:
    print(path.relative_to(project_dir))

zip_path = output_dir.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in output_files:
        zf.write(path, arcname=path.relative_to(output_dir.parent))

print("Training results archived")
print("zip_path =", zip_path)
